# 2. The Factors

### Data
Use the data found in `data/factor_pricing_data.xlsx`.

Factors: Monthly excess return data for the overall equity market, $\tilde{r}^{\text{MKT}}$.

* The column header to the market factor is `MKT` rather than `MKT-RF`, but it is indeed already in excess return form.

* The sheet also contains data on five additional factors.

* All factor data is already provided as excess returns

### 1. 
Analyze the factors, similar to how you analyzed the three Fama-French factors in `Homework 4`.

You now have three additional factors, so let’s compare their univariate statistics. 

* mean
* volatility    
* Sharpe

In [19]:
import pandas as pd
import numpy as np 
DATA = "data/factor_pricing_data.xlsx"
factors = pd.read_excel(DATA, sheet_name = 'factors (excess returns)', index_col = 'Date', parse_dates = True)

after_2015 = factors.loc['2015-01':]
factor_names = ['MKT', 'SMB', 'HML', 'RMW', 'CMA', 'UMD']

stats = []
periods = {
    "full sample": factors,
    "2015 to present": after_2015
}
for per_name, period in periods.items():
    for factor in factor_names:
        mean = period[factor].mean() * 12
        vol = period[factor].std() * np.sqrt(12)
        sharpe = mean / vol
        stats.append({
            'Period': per_name,
            'Factor': factor,
            'Mean': mean,
            'Volatility': vol,
            'Sharpe Ratio': sharpe
        })

stats_df = pd.DataFrame(stats)
stats_df


,Period,Factor,Mean,Volatility,Sharpe Ratio
0,full sample,MKT,0.084944,0.156374,0.543210
1,full sample,SMB,0.005420,0.101365,0.053469
2,full sample,HML,0.026508,0.108907,0.243402
3,full sample,RMW,0.045139,0.083055,0.543484
4,full sample,CMA,0.027686,0.072496,0.381900
5,full sample,UMD,0.061769,0.153524,0.402343
6,2015 to present,MKT,0.107419,0.158562,0.677460
7,2015 to present,SMB,-0.027813,0.103531,-0.268644
8,2015 to present,HML,-0.015610,0.130814,-0.119327
9,2015 to present,RMW,0.044671,0.072908,0.612702


### 2. 

Based on the factor statistics above, answer the following.
* Does each factor have a positive risk premium (positive expected excess return)? 
* How have the factors performed since the time of the case, (2015-present)?


For the full sample, each factor has a positive risk premium (positive expected excess return) that ranges from approximately 0.027 to 0.08. If I were to look at the subsample that only includes data from 2015 onwards, I notice that the expected excess returns of SMB, HML, and CMA are all negative. This tells me tha the premiums of size, value, and investment have dissapeared since the time of this case. MKT, RMW, and UMD all remained positive, with MKT increasing in expected excess return when looking at the 2015 and onwards subsample, RMW having roughly the same risk premium, and UMD having a smaller positive risk premium in the 2015 subsample.

### 3. 

Report the correlation matrix across the six factors.
* Does the construction method succeed in keeping correlations small?
* Fama and French say that HML is somewhat redundant in their 5-factor model. Does this seem to be the case?

In [21]:

corr_matrix = factors[['MKT', 'SMB', 'HML', 'RMW', 'CMA', 'UMD']].corr()
print(corr_matrix)

          MKT       SMB       HML       RMW       CMA       UMD
MKT  1.000000  0.228076 -0.206256 -0.249761 -0.351911 -0.178738
SMB  0.228076  1.000000 -0.027296 -0.412657 -0.053969 -0.057655
HML -0.206256 -0.027296  1.000000  0.220277  0.680753 -0.214139
RMW -0.249761 -0.412657  0.220277  1.000000  0.140033  0.074195
CMA -0.351911 -0.053969  0.680753  0.140033  1.000000  0.000709
UMD -0.178738 -0.057655 -0.214139  0.074195  0.000709  1.000000


Looking at the correlation matrix between the 6 factors, I would say that the construction method does generally succeed in keeping correlations small. The only pair that I would keep an eye on is HML and CMA which has a correlation coefficient of 0.68 which tells me they are highly, positively correlated. I would say that HML is redundant with CMA since I just pointed out that they have a  strong relationship.


### 4. 

Report the tangency weights for a portfolio of these 6 factors.
* Which factors seem most important? And Least?
* Are the factors with low mean returns still useful?
* Re-do the tangency portfolio, but this time only include MKT, SMB, HML, and UMD. Which factors get high/low tangency weights now?


In [24]:
def calc_tangency_weights(excess_returns):
    mu = excess_returns.mean()
    cov_matrix = excess_returns.cov()
    inv_cov_times_mu = np.linalg.solve(cov_matrix, mu)
    tan_w = inv_cov_times_mu / inv_cov_times_mu.sum()
    return pd.Series(tan_w, index = excess_returns.columns)
tan_w = calc_tangency_weights(factors[factor_names])
tan_w.round(4)

MKT    0.2161
SMB    0.0667
HML   -0.0197
RMW    0.3059
CMA    0.3166
UMD    0.1144
dtype: float64

RMW and CMA get the higher weights of 0.31 and 0.32, respectively, so they are the factors that seem most important. SMB has the lowest positive weight at 0.07 so that seems like one of the least important, and HML has a near zero weighting of -0.0197 so that is the least important. But, HML is not the least important because it's negative, but because it is the closest to zero.

Yes, factors with low mean returns are still useful. RMW only had an expected excess return of 0.045, but it still has one of the highest weightings (0.31). Similarly, CMA had a low expected excess return of 0.028 and still had one of the highest weightings (0.32). 

In [25]:
factors_4 = ['MKT', 'SMB', 'HML', 'UMD']
tan_w4 = calc_tangency_weights(factors[factors_4])
tan_w4.round(4)

MKT    0.3715
SMB   -0.0537
HML    0.3679
UMD    0.3143
dtype: float64

If I only include MKT, SMB, HML, and UMD, then the factors that get high weights are MKT, HML, and UMD so they're all pretty important. But SMB has the weight closest to zero, -0.0537, therefore it is the least important. 


What do you conclude about the importance or unimportance of these styles?

I conclude that CMA and HML are highly correlated, which reinforces my previous statement in 1.3, since in the full portfolio with all 6 factors, HML was the least important because it had the smallest weighting. But, when I removed CMA from the portfolio, HML's weighting jumped to 0.37 because there is no longer an asset in the portfolio that HML is highly correlated with.

***

# 3. Testing Modern LPMs

Consider the following factor models:
* CAPM: MKT
* Fama-French 3F: MKT, SMB, HML
* Fama-French 5F: MKT, SMB, HML, RMW, CMA
* AQR: MKT, HML, RMW, UMD

Our labeling of the last model as the **AQR** is just for concreteness. The firm is well-known for these factors and an unused case study discusses that further.

For instance, for the AQR model is...

$$
\mathbb{E}[\tilde{r}^i] 
= \beta^{i,\mathrm{MKT}} \, \mathbb{E}[\tilde{f}^{\mathrm{MKT}}] 
+ \beta^{i,\mathrm{HML}} \, \mathbb{E}[\tilde{f}^{\mathrm{HML}}] 
+ \beta^{i,\mathrm{RMW}} \, \mathbb{E}[\tilde{f}^{\mathrm{RMW}}] 
+ \beta^{i,\mathrm{UMD}} \, \mathbb{E}[\tilde{f}^{\mathrm{UMD}}]
$$

We will test these models with the time-series regressions. Namely, for each asset i, estimate the following regression to test the AQR model:

$$
\tilde{r}^i_t 
= \alpha^i 
+ \beta^{i,\mathrm{MKT}} \tilde{f}^{\mathrm{MKT}}_t 
+ \beta^{i,\mathrm{HML}} \tilde{f}^{\mathrm{HML}}_t 
+ \beta^{i,\mathrm{RMW}} \tilde{f}^{\mathrm{RMW}}_t 
+ \beta^{i,\mathrm{UMD}} \tilde{f}^{\mathrm{UMD}}_t 
+ \varepsilon_t
$$

### Data

* Monthly excess return data on `n=49` equity portfolios sorted by their industry. Denote these as $\tilde{r}^i$ , for $i = 1, \dots, 49$.

* You do NOT need the risk-free rate data. It is provided only for completeness. The other two tabs are already in terms of excess returns.

### 1. 

Test the AQR 4-Factor Model using time-series regressions.

For each regression, report the estimated α and r-squared.


### 2. 

Calculate the mean-absolute-error of the estimated alphas.

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^n|\tilde{\alpha}^i|$$

* If the pricing model worked, should these alpha estimates be large or small? Why?

* Based on your MAE stat, does this seem to support the pricing model or not?



### 3. 

Test the CAPM, FF 3-Factor Model and the FF 5-Factor Model.
   * Report the MAE statistic for each of these models and compare it with the AQR Model MAE.
   * Which model fits best?
   

### 4. 

Does any particular factor seem especially important or unimportant for pricing? Do you think Fama and French should use the Momentum Factor?

### 5. 

This does not matter for pricing, but report the average (across $n$ estimations) of the time-series regression r-squared statistics.
   * Do this for each of the four models you tested.
   * Do these models lead to high time-series r-squared stats? That is, would these factors be good in a Linear Factor Decomposition of the assets?


### 6. 

Re-test all four models using cross-sectional regressions with an intercept.

* Report the time-series premia of the factors (just their sample averages,) and compare to the cross-sectionally estimated premia of the factors. Do they differ substantially?
* Report the MAE of the cross-sectional regression residuals for each of the four models. How do they compare to the MAE of the time-series alphas?



In Homework 4 (25 size/value portfolios, January 1981 through August 2026), the sample mean market excess return was positive. The cross-sectional CAPM market premium was negative with an intercept and positive when the intercept was fixed at zero.